# Tuned XGBoost → GBDT Leaf Embedding → MLP

This notebook uses **your tuned XGBoost search setup** as the GBDT/leaf-embedding generator.

Pipeline:

`Official encoded train/test → leakage-safe features → five XGBoost tuning methods → best tuned XGBoost → leaf indices → one-hot leaf embedding → concatenate with original features → random projection → PCA → MLP → evaluation → save model components`

The test set is kept for final evaluation and is not used to select hyperparameters.


In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import time
import random
import warnings
import joblib

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from scipy.sparse import hstack, csr_matrix, issparse

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Libraries loaded.")

# sklearn compatibility: sparse_output was introduced in newer sklearn versions.
def make_sparse_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


Libraries loaded.


In [2]:
# ============================================================
# CHECKPOINT / RESUME UTILITIES
# ============================================================
# Completed stages are saved to disk so later errors do not
# require repeating earlier work.

CHECKPOINT_DIR = Path(
    r"E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline"
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def save_checkpoint(obj, filename):
    path = CHECKPOINT_DIR / filename
    joblib.dump(obj, path)
    print(f"Checkpoint saved: {path}")

def load_checkpoint(filename):
    path = CHECKPOINT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    obj = joblib.load(path)
    print(f"Checkpoint loaded: {path}")
    return obj

def checkpoint_exists(filename):
    return (CHECKPOINT_DIR / filename).exists()

print("Checkpoint directory:", CHECKPOINT_DIR)


Checkpoint directory: E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline


## Checkpoint / Resume

Checkpoints are saved under:

`E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline`

one file per completed stage:

- `02_xgb_random_result.pkl`
- `03_xgb_grid_result.pkl`
- `04_xgb_bayes_result.pkl`
- `05_xgb_hyperopt_result.pkl`
- `06_xgb_optuna_result.pkl`
- `FINAL_all_model_components.pkl` / `FINAL_metrics.csv`

**The five-search cell is now resumable.** If the notebook (or kernel) crashes partway
through, just re-run the notebook from the top: any stage whose checkpoint file already
exists on disk is loaded from the `.pkl` instead of being recomputed, and only the
stage(s) that never finished actually run again.

**Root cause of the `KeyError: 'xgb_ne'` crash:** `hyperopt`'s `fmin()` return value is
keyed by the label strings passed into `hp.*` calls, but the dict handed to your
objective function on every trial is keyed by the *search-space dict's own keys*
(`n_estimators`, `max_depth`, ...). The old code used one `converter` for both, which
only works for the dict-keys case. The fix stops relying on `fmin`'s return value
entirely -- each trial stores its own already-converted params inside the trial result,
and the winner is read back from `trials.best_trial`.

For Hyperopt, the random-state line remains:

```python
rstate=np.random.default_rng(SEED)
```


In [3]:
# ============================================================
# 2. LOAD OFFICIAL ENCODED TRAIN / TEST FILES
# ============================================================

TRAIN_FILE = Path(r"E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_train.csv")
TEST_FILE = Path(r"E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_test.csv")

if not TRAIN_FILE.exists():
    raise FileNotFoundError(f"Train file not found: {TRAIN_FILE.resolve()}")
if not TEST_FILE.exists():
    raise FileNotFoundError(f"Test file not found: {TEST_FILE.resolve()}")

train = pd.read_csv(TRAIN_FILE)
test = pd.read_csv(TEST_FILE)

print("Train shape:", train.shape)
print("Test shape :", test.shape)
display(train.head())


Train shape: (16000, 81)
Test shape : (4000, 81)


,id,goal_usd,log_goal_usd,duration_days,prelaunch_days,name_char_length,name_word_count,blurb_char_length,blurb_word_count,launch_year,...,location_type_County,location_type_Island,location_type_LocalAdmin,location_type_Miscellaneous,location_type_Suburb,location_type_Town,location_type_Unknown,location_type_Zip,target_usd,log_target
0,160053502,1490.1241,7.307286,30.000000,13.537003,30.0,5.0,25.0,5.0,2020.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2694.360622,7.899287
1,1797462698,5000.0000,8.517393,35.041668,18.665070,8.0,2.0,108.0,18.0,2017.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,122.000000,4.812184
2,1582340481,10000.0000,9.210441,30.000000,10.058495,53.0,7.0,108.0,13.0,2025.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.000000,0.000000
3,498571554,15475.7350,9.647093,29.958334,7.044641,39.0,7.0,110.0,18.0,2022.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,26102.038065,10.169807
4,1914908476,1400.0000,7.244942,60.000000,13.044236,29.0,5.0,125.0,24.0,2025.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1478.000000,7.299121


In [4]:
# ============================================================
# 3. TARGETS AND LEAKAGE-SAFE FEATURES
# ============================================================

TARGET = "target_usd"
LOG_TARGET = "log_target"
ID_COL = "id"

assert TARGET in train.columns and LOG_TARGET in train.columns
assert TARGET in test.columns and LOG_TARGET in test.columns

# These are the only columns automatically excluded here.
# Add any other post-launch / target-derived columns to DROP_COLS
# if your dataset contains them.
DROP_COLS = [ID_COL]
EXCLUDE_COLS = DROP_COLS + [TARGET, LOG_TARGET]

feature_cols = [c for c in train.columns if c not in EXCLUDE_COLS]

# Hard leakage checks
assert TARGET not in feature_cols
assert LOG_TARGET not in feature_cols
assert ID_COL not in feature_cols

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y_train = train[LOG_TARGET].copy()
y_test = test[LOG_TARGET].copy()
actual_usd = test[TARGET].copy()

print("Input features:", len(feature_cols))
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Target :", LOG_TARGET)


Input features: 78
X_train: (16000, 78)
X_test : (4000, 78)
Target : log_target


In [5]:
# ============================================================
# 4. CHECK THAT OFFICIAL FILES ARE NUMERIC / ENCODED
# ============================================================

non_numeric = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

if non_numeric:
    raise TypeError(
        "The supplied ML_train.csv contains non-numeric input columns: "
        + str(non_numeric)
        + "\nThis notebook expects the official encoded train/test files, "
        "as in your existing baseline/tuning pipeline."
    )

print("All model input columns are numeric.")


All model input columns are numeric.


In [6]:
# ============================================================
# 5. ROBUST PREPROCESSING FOR THE MLP REPRESENTATION
# ============================================================

# Your official files are already encoded, so there is no need to
# one-hot encode them again. We only impute and robust-scale.

robust_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])

X_train_pre = robust_preprocessor.fit_transform(X_train)
X_test_pre = robust_preprocessor.transform(X_test)

# Convert to dense for the later random projection/PCA stages.
if issparse(X_train_pre):
    X_train_pre = X_train_pre.toarray()
    X_test_pre = X_test_pre.toarray()

# Bounded nonlinear clipping for the neural representation.
def smooth_clip(X, limit=5.0):
    return np.tanh(X / limit) * limit

X_train_pre = smooth_clip(X_train_pre)
X_test_pre = smooth_clip(X_test_pre)

print("Preprocessed train:", X_train_pre.shape)
print("Preprocessed test :", X_test_pre.shape)


Preprocessed train: (16000, 78)
Preprocessed test : (4000, 78)


In [7]:
# ============================================================
# 6. YOUR EXACT XGBOOST BASE ESTIMATOR + SEARCH SPACES
# ============================================================

xgb_base = XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)

xgb_grid = {
    'n_estimators':[500,800],
    'max_depth':[4,7],
    'learning_rate':[0.03,0.05],
    'min_child_weight':[1,3],
    'subsample':[0.8,1.0],
    'colsample_bytree':[0.8,1.0]
}

xgb_random = {
    'n_estimators':[400,600,800,1000],
    'max_depth':[3,4,5,6,7,8],
    'learning_rate':[.01,.02,.03,.05,.08],
    'min_child_weight':[1,3,5,8],
    'subsample':[.7,.8,.9,1.0],
    'colsample_bytree':[.7,.8,.9,1.0],
    'reg_alpha':[0,.01,.05,.1],
    'reg_lambda':[1,2,5]
}

from skopt.space import Integer, Real
xgb_bayes = {
    'n_estimators':Integer(400,1200),
    'max_depth':Integer(3,10),
    'learning_rate':Real(.01,.1,prior='log-uniform'),
    'min_child_weight':Integer(1,10),
    'subsample':Real(.7,1.0),
    'colsample_bytree':Real(.7,1.0),
    'reg_alpha':Real(1e-4,1.0,prior='log-uniform'),
    'reg_lambda':Real(1.0,10.0,prior='log-uniform')
}

from hyperopt import hp
# NOTE: the label passed as the 1st argument to each hp.* call is only
# used internally by hyperopt for bookkeeping. The dict KEY on the left
# ('n_estimators', 'max_depth', ...) is what your objective function
# actually receives at runtime. Labels are named the same as the keys
# here on purpose, to avoid the KeyError trap from the previous version.
xgb_hyper = {
    'n_estimators':hp.quniform('n_estimators',400,1200,50),
    'max_depth':hp.quniform('max_depth',3,10,1),
    'learning_rate':hp.loguniform('learning_rate',np.log(.01),np.log(.1)),
    'min_child_weight':hp.quniform('min_child_weight',1,10,1),
    'subsample':hp.uniform('subsample',.7,1),
    'colsample_bytree':hp.uniform('colsample_bytree',.7,1),
    'reg_alpha':hp.loguniform('reg_alpha',np.log(1e-4),np.log(1)),
    'reg_lambda':hp.loguniform('reg_lambda',np.log(1),np.log(10))
}

print("Search spaces loaded.")


Search spaces loaded.


In [8]:
# ============================================================
# 7. TUNING SETTINGS
# ============================================================

# Keep these consistent with your existing tuning notebook if
# you already have preferred values.
CV_FOLDS = 5
RANDOM_SEARCH_ITER = 30
HYPEROPT_MAX_EVALS = 30
OPTUNA_N_TRIALS = 30

cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

# Negative MSE is used by sklearn search APIs because they maximize
# scoring. We convert it back to positive MSE/RMSE when reporting.
SCORING = 'neg_mean_squared_error'

print(f"CV folds: {CV_FOLDS}")
print(f"Random search iterations: {RANDOM_SEARCH_ITER}")
print(f"Hyperopt evaluations: {HYPEROPT_MAX_EVALS}")
print(f"Optuna trials: {OPTUNA_N_TRIALS}")


CV folds: 5
Random search iterations: 30
Hyperopt evaluations: 30
Optuna trials: 30


In [9]:
# ============================================================
# 8. RANDOM SEARCH HELPER
# ============================================================

def tune_random(name, estimator, param_grid, X, y):
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_grid,
        n_iter=RANDOM_SEARCH_ITER,
        scoring=SCORING,
        cv=cv,
        refit=True,
        random_state=SEED,
        n_jobs=-1,
        verbose=1
    )
    start = time.time()
    search.fit(X, y)
    elapsed = time.time() - start
    best_mse = -search.best_score_
    return {
        'Method': 'Random Search',
        'Estimator': search.best_estimator_,
        'Best_CV_MSE_log': best_mse,
        'Best_CV_RMSE_log': np.sqrt(best_mse),
        'Best_Params': search.best_params_,
        'Time_Seconds': elapsed
    }

print("Random Search helper ready.")


Random Search helper ready.


In [10]:
# ============================================================
# 9. GRID SEARCH HELPER
# ============================================================

def tune_grid(name, estimator, param_grid, X, y):
    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=SCORING,
        cv=cv,
        refit=True,
        n_jobs=-1,
        verbose=1
    )
    start = time.time()
    search.fit(X, y)
    elapsed = time.time() - start
    best_mse = -search.best_score_
    return {
        'Method': 'Grid Search',
        'Estimator': search.best_estimator_,
        'Best_CV_MSE_log': best_mse,
        'Best_CV_RMSE_log': np.sqrt(best_mse),
        'Best_Params': search.best_params_,
        'Time_Seconds': elapsed
    }

print("Grid Search helper ready.")


Grid Search helper ready.


In [11]:
# ============================================================
# 10. BAYESIAN SEARCH HELPER
# ============================================================

from skopt import BayesSearchCV

def tune_bayes(name, estimator, search_spaces, X, y):
    search = BayesSearchCV(
        estimator=estimator,
        search_spaces=search_spaces,
        n_iter=30,
        scoring=SCORING,
        cv=cv,
        refit=True,
        random_state=SEED,
        n_jobs=-1,
        verbose=0
    )
    start = time.time()
    search.fit(X, y)
    elapsed = time.time() - start
    best_mse = -search.best_score_
    return {
        'Method': 'Bayesian Search',
        'Estimator': search.best_estimator_,
        'Best_CV_MSE_log': best_mse,
        'Best_CV_RMSE_log': np.sqrt(best_mse),
        'Best_Params': search.best_params_,
        'Time_Seconds': elapsed
    }

print("Bayesian Search helper ready.")


Bayesian Search helper ready.


In [12]:
# ============================================================
# 11. HYPEROPT HELPER  (FIXED)
# ============================================================
# Bug that caused "KeyError: 'xgb_ne'":
#   fmin()'s return value (the argmin) is keyed by the *label* strings
#   passed into hp.* (e.g. 'xgb_ne'), but the dict handed to objective()
#   on every trial is keyed by the SPACE DICT'S OWN KEYS
#   ('n_estimators', 'max_depth', ...). The old code ran the same
#   `converter` on both, so anything expecting label-style keys blew up.
#
# Fix: never read fmin's return value at all. Each trial stores its own
# already-converted params inside the trial result, and the winner is
# read back from trials.best_trial — so there's only ever one place the
# key names have to line up: your `converter`, against the SPACE DICT
# keys (i.e. use converter like `p['n_estimators']`, not `p['xgb_ne']`).

from hyperopt import fmin, tpe, Trials, STATUS_OK

def tune_hyperopt(name, estimator, space, converter, X, y, max_evals):
    # Hyperopt objective uses fixed K-fold CV on training data only.
    X_np = X.values if isinstance(X, pd.DataFrame) else X
    y_np = np.asarray(y)

    def objective(raw_params):
        # raw_params is keyed by the SPACE DICT's keys, e.g. 'n_estimators'
        params = converter(raw_params)
        fold_mse = []

        for tr_idx, va_idx in cv.split(X_np):
            model = XGBRegressor(
                objective='reg:squarederror',
                eval_metric='rmse',
                random_state=SEED,
                n_jobs=-1,
                tree_method='hist',
                **params
            )
            model.fit(X_np[tr_idx], y_np[tr_idx], verbose=False)
            pred = model.predict(X_np[va_idx])
            fold_mse.append(mean_squared_error(y_np[va_idx], pred))

        loss = float(np.mean(fold_mse))
        # Stash the already-converted params on the trial result itself,
        # so we never have to re-derive them from fmin's differently-keyed
        # return value.
        return {'loss': loss, 'status': STATUS_OK, 'params': params}

    trials = Trials()
    start = time.time()
    fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        rstate=np.random.default_rng(SEED),
        show_progressbar=True
    )
    elapsed = time.time() - start

    best_trial_result = trials.best_trial['result']
    best_params = best_trial_result['params']
    best_mse = best_trial_result['loss']

    best_model = XGBRegressor(
        objective='reg:squarederror',
        eval_metric='rmse',
        random_state=SEED,
        n_jobs=-1,
        tree_method='hist',
        **best_params
    )
    best_model.fit(X, y, verbose=False)

    return {
        'Method': 'Hyperopt',
        'Estimator': best_model,
        'Best_CV_MSE_log': float(best_mse),
        'Best_CV_RMSE_log': float(np.sqrt(best_mse)),
        'Best_Params': best_params,
        'Time_Seconds': elapsed
    }

print("Hyperopt helper ready.")

Hyperopt helper ready.


In [13]:
# ============================================================
# 12. OPTUNA HELPER
# ============================================================

def tune_optuna(name, estimator, param_func, X, y, n_trials):
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    X_np = X.values if isinstance(X, pd.DataFrame) else X
    y_np = np.asarray(y)

    def objective(trial):
        params = param_func(trial)
        fold_mse = []

        for tr_idx, va_idx in cv.split(X_np):
            model = XGBRegressor(
                objective='reg:squarederror',
                eval_metric='rmse',
                random_state=SEED,
                n_jobs=-1,
                tree_method='hist',
                **params
            )
            model.fit(X_np[tr_idx], y_np[tr_idx], verbose=False)
            pred = model.predict(X_np[va_idx])
            fold_mse.append(mean_squared_error(y_np[va_idx], pred))

        return float(np.mean(fold_mse))

    start = time.time()
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    elapsed = time.time() - start

    best_params = study.best_params
    best_model = XGBRegressor(
        objective='reg:squarederror',
        eval_metric='rmse',
        random_state=SEED,
        n_jobs=-1,
        tree_method='hist',
        **best_params
    )
    best_model.fit(X, y, verbose=False)

    return {
        'Method': 'Optuna',
        'Estimator': best_model,
        'Best_CV_MSE_log': float(study.best_value),
        'Best_CV_RMSE_log': float(np.sqrt(study.best_value)),
        'Best_Params': best_params,
        'Time_Seconds': elapsed
    }

print("Optuna helper ready.")


Optuna helper ready.


In [14]:
# ============================================================
# 13. COMPARE FIVE SEARCH METHODS
# ============================================================

def compare_five(results):
    rows = []
    for r in results:
        rows.append({
            'Method': r['Method'],
            'Best_CV_MSE_log': r['Best_CV_MSE_log'],
            'Best_CV_RMSE_log': r['Best_CV_RMSE_log'],
            'Time_Seconds': r.get('Time_Seconds', np.nan),
            'Best_Params': r['Best_Params']
        })
    return pd.DataFrame(rows).sort_values('Best_CV_MSE_log').reset_index(drop=True)

print("Comparison helper ready.")


Comparison helper ready.


In [15]:
# ============================================================
# 7A. VERIFY TUNING CALL SIGNATURES / DEPENDENCIES
# ============================================================

import inspect

print('tune_hyperopt signature:', inspect.signature(tune_hyperopt) if 'tune_hyperopt' in globals() else 'defined below')
print('Required tuning flow:')
print('tune_random(name, estimator, param_grid, X, y)')
print('tune_grid(name, estimator, param_grid, X, y)')
print('tune_bayes(name, estimator, search_spaces, X, y)')
print('tune_hyperopt(name, estimator, space, converter, X, y, max_evals)')
print('tune_optuna(name, estimator, param_func, X, y, n_trials)')


tune_hyperopt signature: (name, estimator, space, converter, X, y, max_evals)
Required tuning flow:
tune_random(name, estimator, param_grid, X, y)
tune_grid(name, estimator, param_grid, X, y)
tune_bayes(name, estimator, search_spaces, X, y)
tune_hyperopt(name, estimator, space, converter, X, y, max_evals)
tune_optuna(name, estimator, param_func, X, y, n_trials)


In [16]:
# ============================================================
# 14. RUN YOUR FIVE XGBOOST SEARCH METHODS (RESUMABLE)
# ============================================================
# Each of the 5 searches is independently checkpointed. Re-running this
# cell after a crash/interrupt will SKIP any search whose checkpoint file
# already exists on disk and load its saved result instead -- only the
# searches that never finished actually run again.

# --- 1/5 Random Search -------------------------------------------------
if checkpoint_exists("02_xgb_random_result.pkl"):
    print("1/5 Random Search -> found checkpoint, loading")
    xgb_random_result = load_checkpoint("02_xgb_random_result.pkl")
else:
    print("1/5 Random Search -> running")
    xgb_random_result = tune_random('XGBoost', xgb_base, xgb_random, X_train, y_train)
    save_checkpoint(xgb_random_result, "02_xgb_random_result.pkl")

# --- 2/5 Grid Search -----------------------------------------------------
if checkpoint_exists("03_xgb_grid_result.pkl"):
    print("2/5 Grid Search -> found checkpoint, loading")
    xgb_grid_result = load_checkpoint("03_xgb_grid_result.pkl")
else:
    print("2/5 Grid Search -> running")
    xgb_grid_result = tune_grid('XGBoost', xgb_base, xgb_grid, X_train, y_train)
    save_checkpoint(xgb_grid_result, "03_xgb_grid_result.pkl")

# --- 3/5 Bayesian Search --------------------------------------------------
if checkpoint_exists("04_xgb_bayes_result.pkl"):
    print("3/5 Bayesian Search -> found checkpoint, loading")
    xgb_bayes_result = load_checkpoint("04_xgb_bayes_result.pkl")
else:
    print("3/5 Bayesian Search -> running")
    xgb_bayes_result = tune_bayes('XGBoost', xgb_base, xgb_bayes, X_train, y_train)
    save_checkpoint(xgb_bayes_result, "04_xgb_bayes_result.pkl")

# --- 4/5 Hyperopt ------------------------------------------------------------
# IMPORTANT: this converter's keys must match xgb_hyper's own dict keys
# ('n_estimators', 'max_depth', ...) -- NOT the hp.* label strings. This is
# what fixes "KeyError: 'xgb_ne'".
if checkpoint_exists("05_xgb_hyperopt_result.pkl"):
    print("4/5 Hyperopt -> found checkpoint, loading")
    xgb_hyper_result = load_checkpoint("05_xgb_hyperopt_result.pkl")
else:
    print("4/5 Hyperopt -> running")
    xgb_hyper_result = tune_hyperopt(
        'XGBoost', xgb_base, xgb_hyper,
        lambda p: {
            'n_estimators': int(p['n_estimators']),
            'max_depth': int(p['max_depth']),
            'learning_rate': float(p['learning_rate']),
            'min_child_weight': int(p['min_child_weight']),
            'subsample': float(p['subsample']),
            'colsample_bytree': float(p['colsample_bytree']),
            'reg_alpha': float(p['reg_alpha']),
            'reg_lambda': float(p['reg_lambda'])
        },
        X_train, y_train, HYPEROPT_MAX_EVALS
    )
    save_checkpoint(xgb_hyper_result, "05_xgb_hyperopt_result.pkl")

# --- 5/5 Optuna ------------------------------------------------------------
if checkpoint_exists("06_xgb_optuna_result.pkl"):
    print("5/5 Optuna -> found checkpoint, loading")
    xgb_optuna_result = load_checkpoint("06_xgb_optuna_result.pkl")
else:
    print("5/5 Optuna -> running")
    xgb_optuna_result = tune_optuna(
        'XGBoost', xgb_base,
        lambda t: {
            'n_estimators': t.suggest_int('n_estimators', 400, 1200, step=50),
            'max_depth': t.suggest_int('max_depth', 3, 10),
            'learning_rate': t.suggest_float('learning_rate', .01, .1, log=True),
            'min_child_weight': t.suggest_int('min_child_weight', 1, 10),
            'subsample': t.suggest_float('subsample', .7, 1.0),
            'colsample_bytree': t.suggest_float('colsample_bytree', .7, 1.0),
            'reg_alpha': t.suggest_float('reg_alpha', 1e-4, 1, log=True),
            'reg_lambda': t.suggest_float('reg_lambda', 1, 10, log=True)
        },
        X_train, y_train, OPTUNA_N_TRIALS
    )
    save_checkpoint(xgb_optuna_result, "06_xgb_optuna_result.pkl")

xgb_results = [
    xgb_random_result,
    xgb_grid_result,
    xgb_bayes_result,
    xgb_hyper_result,
    xgb_optuna_result
]

print("All five searches completed (checkpointed results reused where available).")

1/5 Random Search -> running
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Checkpoint saved: E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\02_xgb_random_result.pkl
2/5 Grid Search -> running
Fitting 5 folds for each of 64 candidates, totalling 320 fits
Checkpoint saved: E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\03_xgb_grid_result.pkl
3/5 Bayesian Search -> running
Checkpoint saved: E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\04_xgb_bayes_result.pkl
4/5 Hyperopt -> running
100%|██████████| 30/30 [06:59<00:00, 13.99s/trial, best loss: 5.031765776245267]
Checkpoint saved: E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\05_xgb_hyperopt_result.pkl
5/5 Optuna -> running
Checkpoint saved: E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\06_xgb_optuna_result.pkl
All five searches completed (checkpointed results reused where available).


In [17]:
# ============================================================
# 15. SELECT BEST TUNED XGBOOST
# ============================================================

xgb_tuning_table = compare_five(xgb_results)
display(xgb_tuning_table)

xgb_winner = min(xgb_results, key=lambda r: r['Best_CV_MSE_log'])
xgb_model = xgb_winner['Estimator']
xgb_best_method = xgb_winner['Method']

print('XGBoost winner:', xgb_best_method)
print('CV MSE_log:', xgb_winner['Best_CV_MSE_log'])
print('CV RMSE_log:', xgb_winner['Best_CV_RMSE_log'])
print('Best parameters:')
print(xgb_winner['Best_Params'])


,Method,Best_CV_MSE_log,Best_CV_RMSE_log,Time_Seconds,Best_Params
0,Bayesian Search,5.024770,2.241600,306.509376,"{'colsample_bytree': 0.879842775722417, 'learn..."
1,Optuna,5.025360,2.241731,342.463845,"{'n_estimators': 1150, 'max_depth': 6, 'learni..."
2,Hyperopt,5.031766,2.243160,419.596558,"{'n_estimators': 950, 'max_depth': 7, 'learnin..."
3,Grid Search,5.040665,2.245143,240.947688,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."
4,Random Search,5.043548,2.245785,125.838290,"{'subsample': 0.9, 'reg_lambda': 5, 'reg_alpha..."


XGBoost winner: Bayesian Search
CV MSE_log: 5.024769810304119
CV RMSE_log: 2.2415998327766085
Best parameters:
OrderedDict({'colsample_bytree': 0.879842775722417, 'learning_rate': 0.013552312824520015, 'max_depth': 7, 'min_child_weight': 5, 'n_estimators': 1200, 'reg_alpha': 0.00017817841992213808, 'reg_lambda': 1.810873447505085, 'subsample': 0.8087154228600544})


In [18]:
# ============================================================
# 16. FINAL FIT OF WINNING TUNED XGBOOST ON TRAINING DATA
# ============================================================

# Search has already used CV on the training set. The selected
# estimator is now fitted on all official training rows.
# The test set is NOT used for fitting or selection.

start_time = time.time()
xgb_model.fit(X_train, y_train, verbose=False)
xgb_fit_time = time.time() - start_time

print(f"Selected tuned XGBoost fitted in {xgb_fit_time:.2f} seconds")
print(xgb_model)


Selected tuned XGBoost fitted in 3.73 seconds
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.879842775722417, device=None,
             early_stopping_rounds=None, enable_categorical=True,
             eval_metric='rmse', feature_types=None, feature_weights=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.013552312824520015,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=7, max_leaves=None,
             min_child_weight=5, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1200, n_jobs=-1,
             num_parallel_tree=None, ...)


In [19]:
# ============================================================
# 17. EVALUATION FUNCTIONS
# ============================================================

def evaluate_predictions(y_true_log, y_pred_log, y_true_usd, y_pred_usd, training_time=None):
    mse_log = mean_squared_error(y_true_log, y_pred_log)
    rmse_log = np.sqrt(mse_log)
    mae_log = mean_absolute_error(y_true_log, y_pred_log)
    r2_log = r2_score(y_true_log, y_pred_log)

    mse_usd = mean_squared_error(y_true_usd, y_pred_usd)
    rmse_usd = np.sqrt(mse_usd)
    mae_usd = mean_absolute_error(y_true_usd, y_pred_usd)
    r2_usd = r2_score(y_true_usd, y_pred_usd)

    rmsle = np.sqrt(np.mean((
        np.log1p(np.maximum(y_true_usd, 0)) -
        np.log1p(np.maximum(y_pred_usd, 0))
    ) ** 2))

    result = {
        'MAE_log': mae_log,
        'MSE_log': mse_log,
        'RMSE_log': rmse_log,
        'R2_log': r2_log,
        'MAE_USD': mae_usd,
        'MSE_USD': mse_usd,
        'RMSE_USD': rmse_usd,
        'R2_USD': r2_usd,
        'RMSLE': rmsle
    }
    if training_time is not None:
        result['Training_Time_Seconds'] = training_time
    return result

# Direct tuned-XGBoost benchmark
xgb_pred_log = xgb_model.predict(X_test)
xgb_pred_usd = np.maximum(np.expm1(xgb_pred_log), 0.0)

tuned_xgb_metrics = evaluate_predictions(
    y_test, xgb_pred_log, actual_usd, xgb_pred_usd, xgb_fit_time
)

display(pd.DataFrame([tuned_xgb_metrics], index=['Tuned XGBoost']))


,MAE_log,MSE_log,RMSE_log,R2_log,MAE_USD,MSE_USD,RMSE_USD,R2_USD,RMSLE,Training_Time_Seconds
Tuned XGBoost,1.644995,4.999925,2.236051,0.467795,16040.364648,1.109358e+10,105326.062715,0.059985,2.236051,3.733021


In [20]:
# ============================================================
# 18. EXTRACT LEAF INDICES FROM THE TUNED XGBOOST
# ============================================================

leaf_train = xgb_model.apply(X_train)
leaf_test = xgb_model.apply(X_test)

print('Leaf train shape:', leaf_train.shape)
print('Leaf test shape :', leaf_test.shape)
print('First 3 rows of leaf indices:')
print(leaf_train[:3])


Leaf train shape: (16000, 1200)
Leaf test shape : (4000, 1200)
First 3 rows of leaf indices:
[[217. 221. 217. ... 212. 155. 125.]
 [223. 227. 225. ... 133. 149. 125.]
 [222. 226. 224. ... 226. 149. 125.]]


In [21]:
# ============================================================
# 19. ONE-HOT ENCODE THE LEAF INDICES
# ============================================================

leaf_encoder = make_sparse_ohe()

leaf_train_encoded = leaf_encoder.fit_transform(leaf_train)
leaf_test_encoded = leaf_encoder.transform(leaf_test)

print('Leaf embedding train:', leaf_train_encoded.shape)
print('Leaf embedding test :', leaf_test_encoded.shape)


Leaf embedding train: (16000, 99219)
Leaf embedding test : (4000, 99219)


In [22]:
# ============================================================
# 20. CONCATENATE PREPROCESSED FEATURES + LEAF EMBEDDING
# ============================================================

X_train_combined = hstack([
    csr_matrix(X_train_pre),
    leaf_train_encoded
]).tocsr()

X_test_combined = hstack([
    csr_matrix(X_test_pre),
    leaf_test_encoded
]).tocsr()

print('Combined train:', X_train_combined.shape)
print('Combined test :', X_test_combined.shape)


Combined train: (16000, 99297)
Combined test : (4000, 99297)


In [ ]:
import pandas as pd
from pathlib import Path

SAVE_DIR = Path(r"E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline")

feature_names = (
    list(X_train.columns) +
    [f"leaf_{i}" for i in range(leaf_train_encoded.shape[1])]
)

pd.DataFrame.sparse.from_spmatrix(X_train_combined, columns=feature_names).to_csv(
    SAVE_DIR / "X_train_combined.csv", index=False
)
pd.DataFrame.sparse.from_spmatrix(X_test_combined, columns=feature_names).to_csv(
    SAVE_DIR / "X_test_combined.csv", index=False
)

In [23]:
# ============================================================
# 21. RANDOM FEATURE PROJECTION
# ============================================================

rng = np.random.default_rng(SEED)
input_dim = X_train_combined.shape[1]
RANDOM_DIM = 1024

Omega = rng.normal(
    loc=0.0,
    scale=np.sqrt(2.0 / RANDOM_DIM),
    size=(input_dim, RANDOM_DIM)
)

print('Input dimension :', input_dim)
print('Random dimension:', RANDOM_DIM)
print('Projection matrix:', Omega.shape)


Input dimension : 99297
Random dimension: 1024
Projection matrix: (99297, 1024)


In [24]:
# ============================================================
# 22. RANDOM PROJECTION + RELU
# ============================================================

def relu(x):
    return np.maximum(0, x)

X_train_random = relu(X_train_combined @ Omega)
X_test_random = relu(X_test_combined @ Omega)

print('Random train:', X_train_random.shape)
print('Random test :', X_test_random.shape)


Random train: (16000, 1024)
Random test : (4000, 1024)


In [25]:
# ============================================================
# 23. PCA FIXED-SIZE REPRESENTATION
# ============================================================

MAIN_DIM = 128

pca = PCA(n_components=MAIN_DIM, random_state=SEED)
X_train_emb = pca.fit_transform(X_train_random)
X_test_emb = pca.transform(X_test_random)

print('Embedding train:', X_train_emb.shape)
print('Embedding test :', X_test_emb.shape)
print('Explained variance:', pca.explained_variance_ratio_.sum())


Embedding train: (16000, 128)
Embedding test : (4000, 128)
Explained variance: 0.5150654435556905


In [26]:
# ============================================================
# 24. FINAL STANDARDIZATION
# ============================================================

final_scaler = StandardScaler()
X_train_final = final_scaler.fit_transform(X_train_emb)
X_test_final = final_scaler.transform(X_test_emb)

print('Final train:', X_train_final.shape)
print('Final test :', X_test_final.shape)


Final train: (16000, 128)
Final test : (4000, 128)


In [27]:
# ============================================================
# 25. MLP ON TUNED-XGBOOST GBDT REPRESENTATION
# ============================================================

mlp = MLPRegressor(
    hidden_layer_sizes=(256, 256, 128),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    batch_size=256,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=SEED
)

start_time = time.time()
mlp.fit(X_train_final, y_train)
mlp_training_time = time.time() - start_time

print(f'MLP training completed in {mlp_training_time:.2f} seconds')
print('Iterations:', mlp.n_iter_)


MLP training completed in 10.99 seconds
Iterations: 17


In [28]:
# ============================================================
# 26. PREDICTION + EVALUATION
# ============================================================

gbdt_mlp_pred_log = mlp.predict(X_test_final)
gbdt_mlp_pred_usd = np.maximum(np.expm1(gbdt_mlp_pred_log), 0.0)

gbdt_mlp_metrics = evaluate_predictions(
    y_test,
    gbdt_mlp_pred_log,
    actual_usd,
    gbdt_mlp_pred_usd,
    xgb_fit_time + mlp_training_time
)

display(pd.DataFrame([gbdt_mlp_metrics], index=['Tuned XGBoost Leaf Embedding + MLP']))


,MAE_log,MSE_log,RMSE_log,R2_log,MAE_USD,MSE_USD,RMSE_USD,R2_USD,RMSLE,Training_Time_Seconds
Tuned XGBoost Leaf Embedding + MLP,1.831407,5.734729,2.39473,0.389581,18640.43771,1.285152e+10,113364.556978,-0.088974,2.39473,14.719424


In [29]:
# ============================================================
# 27. FINAL COMPARISON
# ============================================================

comparison = pd.DataFrame({
    'Tuned XGBoost': tuned_xgb_metrics,
    'Tuned XGBoost Leaf Embedding + MLP': gbdt_mlp_metrics
})

display(comparison)


,Tuned XGBoost,Tuned XGBoost Leaf Embedding + MLP
MAE_log,1.644995e+00,1.831407e+00
MSE_log,4.999925e+00,5.734729e+00
RMSE_log,2.236051e+00,2.394730e+00
R2_log,4.677953e-01,3.895809e-01
MAE_USD,1.604036e+04,1.864044e+04
MSE_USD,1.109358e+10,1.285152e+10
RMSE_USD,1.053261e+05,1.133646e+05
R2_USD,5.998515e-02,-8.897424e-02
RMSLE,2.236051e+00,2.394730e+00
Training_Time_Seconds,3.733021e+00,1.471942e+01


In [30]:
# ============================================================
# 28. IMPROVEMENT CHECK
# ============================================================

lower_better = ['MAE_log','MSE_log','RMSE_log','MAE_USD','MSE_USD','RMSE_USD','RMSLE']
higher_better = ['R2_log','R2_USD']

print('Percentage improvement of Leaf Embedding + MLP over Tuned XGBoost')
print('Positive means improvement for the selected metric direction.\n')

for metric in lower_better:
    old = tuned_xgb_metrics[metric]
    new = gbdt_mlp_metrics[metric]
    pct = (old - new) / abs(old) * 100
    print(f'{metric:15s}: {pct:+.2f}%')

for metric in higher_better:
    old = tuned_xgb_metrics[metric]
    new = gbdt_mlp_metrics[metric]
    pct = (new - old) / abs(old) * 100 if old != 0 else np.nan
    print(f'{metric:15s}: {pct:+.2f}%')


Percentage improvement of Leaf Embedding + MLP over Tuned XGBoost
Positive means improvement for the selected metric direction.

MAE_log        : -11.33%
MSE_log        : -14.70%
RMSE_log       : -7.10%
MAE_USD        : -16.21%
MSE_USD        : -15.85%
RMSE_USD       : -7.63%
RMSLE          : -7.10%
R2_log         : -16.72%
R2_USD         : -248.33%


In [31]:
# ============================================================
# 29. SAVE FINAL EMBEDDINGS
# ============================================================

OUTPUT_DIR = Path(r'E:\NSU\cse445\EDA attempt3\Dataset\ML dataset')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUTPUT_DIR / 'X_train_TUNED_XGB_GBDT_MLP_embedding.npy', X_train_final)
np.save(OUTPUT_DIR / 'X_test_TUNED_XGB_GBDT_MLP_embedding.npy', X_test_final)

print('Final embeddings saved.')


Final embeddings saved.


In [32]:
# ============================================================
# 30. SAVE MODEL COMPONENTS
# ============================================================

joblib.dump(xgb_model, OUTPUT_DIR / 'tuned_xgb_embedding_model.pkl')
joblib.dump(robust_preprocessor, OUTPUT_DIR / 'tuned_xgb_preprocessor.pkl')
joblib.dump(leaf_encoder, OUTPUT_DIR / 'tuned_xgb_leaf_encoder.pkl')
joblib.dump(Omega, OUTPUT_DIR / 'tuned_xgb_random_projection.npy')
joblib.dump(pca, OUTPUT_DIR / 'tuned_xgb_pca.pkl')
joblib.dump(final_scaler, OUTPUT_DIR / 'tuned_xgb_final_scaler.pkl')
joblib.dump(mlp, OUTPUT_DIR / 'tuned_xgb_gbdt_mlp_model.pkl')

metadata = {
    'seed': SEED,
    'target': TARGET,
    'log_target': LOG_TARGET,
    'best_xgb_method': xgb_best_method,
    'best_cv_mse_log': xgb_winner['Best_CV_MSE_log'],
    'best_cv_rmse_log': xgb_winner['Best_CV_RMSE_log'],
    'best_xgb_params': xgb_winner['Best_Params'],
    'tuned_xgb_test_metrics': tuned_xgb_metrics,
    'gbdt_mlp_test_metrics': gbdt_mlp_metrics,
    'random_projection_dim': RANDOM_DIM,
    'pca_dim': MAIN_DIM,
    'cv_folds': CV_FOLDS,
}
joblib.dump(metadata, OUTPUT_DIR / 'tuned_xgb_gbdt_mlp_metadata.pkl')

print('All model components saved.')
print('Output directory:', OUTPUT_DIR)


All model components saved.
Output directory: E:\NSU\cse445\EDA attempt3\Dataset\ML dataset


## What this notebook is testing

**Benchmark:** your tuned XGBoost selected from Random Search, Grid Search, Bayesian Search, Hyperopt, and Optuna.

**Paper-inspired model:** the winning tuned XGBoost is used as a leaf-index feature extractor; leaf one-hot features are concatenated with the preprocessed original features, projected to a fixed-size representation, and passed to an MLP.

This is **not the full iLTM implementation** because the full paper also contains its meta-trained hypernetwork and retrieval components.


In [34]:
# ============================================================
# FINAL CHECKPOINT: ALL MODEL COMPONENTS + METRICS
# ============================================================

save_checkpoint(
    {
        "xgb_model": xgb_model,
        "preprocessor": robust_preprocessor,
        "leaf_encoder": leaf_encoder,
        "pca": pca,
        "final_scaler": final_scaler,
        "mlp": mlp,
        "metadata": metadata,
    },
    "FINAL_all_model_components.pkl"
)

pd.DataFrame(
    [
        tuned_xgb_metrics,
        gbdt_mlp_metrics
    ],
    index=[
        "Tuned XGBoost",
        "Tuned XGBoost Leaf Embedding + MLP"
    ]
).to_csv(
    CHECKPOINT_DIR / "FINAL_metrics.csv"
)

print("Final model components and metrics saved.")


Checkpoint saved: E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\FINAL_all_model_components.pkl
Final model components and metrics saved.
